# Task 1 -- Data Handling & Memory Management

This notebook demonstrates how the ~19 GB Telecom Italia Milan dataset is loaded and processed under tight memory constraints. The heavy lifting lives in `src/data_loading.py`; here we run it step by step and inspect the evidence.

**Prerequisite:** the raw daily files must be in `data/raw/telecom/` (see `README.md`), or run `python scripts/make_synthetic_data.py` for a runnable sample.

In [1]:
# --- Project setup: make the `src` package importable from anywhere ---
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
print(f"Project root: {_root}")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


Project root: C:\Users\josue\Music\ml_tech_I\formative_1


## 1.V -- Hardware & software environment
The environment determines the memory budget and therefore the data-handling strategy.

In [2]:
from src.utils import print_system_report
_ = print_system_report()

HARDWARE / SOFTWARE ENVIRONMENT
  os                : Windows 11 (10.0.26200)
  machine           : AMD64
  processor         : Intel64 Family 6 Model 197 Stepping 2, GenuineIntel
  cpu logical cores : 16
  total ram         : 31.40 GB
  python            : 3.13.9
  numpy             : 2.3.5
  pandas            : 2.3.3
  tensorflow        : 2.21.0
  gpu               : none (CPU only)


## 1.I -- Discover the raw daily files
The dataset is delivered as one tab-separated file per day. We stream them one at a time so the full volume never has to fit in RAM.

In [3]:
from src.data_loading import discover_raw_files
from src.utils import human_bytes

files = discover_raw_files()
total = sum(f.stat().st_size for f in files)
print(f'{len(files)} daily files | total raw size on disk: {human_bytes(total)}')
files[:3]

62 daily files | total raw size on disk: 19.38 GB


[WindowsPath('C:/Users/josue/Music/ml_tech_I/formative_1/data/raw/telecom/sms-call-internet-mi-2013-11-01.txt'),
 WindowsPath('C:/Users/josue/Music/ml_tech_I/formative_1/data/raw/telecom/sms-call-internet-mi-2013-11-02.txt'),
 WindowsPath('C:/Users/josue/Music/ml_tech_I/formative_1/data/raw/telecom/sms-call-internet-mi-2013-11-03.txt')]

## 1.III -- Memory optimisation: before vs after

We compare a **naive** load (all 8 columns, default int64/float64) against the **optimised** load (only the 3 needed columns, downcast to uint16/float32, country rows aggregated). The table below is the quantitative evidence required by Task 1.III.

In [4]:
from src.data_loading import memory_comparison
mem = memory_comparison(files[0])
mem

Memory comparison on sample file: sms-call-internet-mi-2013-11-01.txt


[timer] naive load: 1.517 s


[timer] optimised load: 1.328 s
                                     rows  columns  memory_bytes memory_human  pct_of_naive
naive (8 cols, int64/float64)     4842625        8     309928132    295.57 MB    100.000000
optimised (3 cols, downcast+agg)  1439982        3      20159880     19.23 MB      6.504695
--> optimised load uses 93.5% less memory than naive.


,rows,columns,memory_bytes,memory_human,pct_of_naive
"naive (8 cols, int64/float64)",4842625,8,309928132,295.57 MB,100.000000
"optimised (3 cols, downcast+agg)",1439982,3,20159880,19.23 MB,6.504695


**Optimisation techniques applied** (Task 1.I / 1.II):
1. **Column pruning** -- `usecols` reads only Square id, Time and Internet traffic, skipping SMS/Call/country-code columns.
2. **Dtype downcasting** -- `square_id` -> uint16, `internet` -> float32 (half the width of the int64/float64 defaults).
3. **Early aggregation** -- per-country rows are summed on load, collapsing each file to one value per (square, 10-min interval).
4. **Columnar Parquet storage** -- the consolidated matrix is written as compressed Parquet, far smaller and faster than TSV.

## 1.I/1.II -- Build the consolidated traffic matrix
Each daily file is streamed, optimised and pivoted to wide form (timestamp x square_id); the days are concatenated into a single float32 matrix and saved to Parquet. On the real data this is the long-running step.

In [5]:
from src.data_loading import (build_traffic_matrix, load_traffic_matrix,
                              compute_area_totals, resolve_target_areas)

try:
    matrix = load_traffic_matrix()   # reuse if already built
    print('Loaded cached traffic_matrix.parquet')
except FileNotFoundError:
    matrix = build_traffic_matrix(files, save=True)

print(matrix.shape)
matrix.iloc[:5, :5]

Loaded cached traffic_matrix.parquet
(8784, 10000)


square_id,1,2,3,4,5
timestamp,,,,,
2013-11-01 00:00:00,11.028366,11.058225,11.090008,10.941881,9.916549
2013-11-01 00:10:00,11.127101,11.167927,11.211384,11.008849,9.987806
2013-11-01 00:20:00,10.892771,10.915638,10.939980,10.826535,9.772990
2013-11-01 00:30:00,8.622424,8.626341,8.630508,8.611082,7.860795
2013-11-01 00:40:00,8.009928,8.020256,8.031251,7.980010,7.263837


In [6]:
totals = compute_area_totals(matrix)
areas = resolve_target_areas(matrix)
print('Three target areas for Tasks 2 & 3:', areas)

Saved area totals -> C:\Users\josue\Music\ml_tech_I\formative_1\data\processed\area_totals.parquet


Three target areas for Tasks 2 & 3: {'highest': 5161, 'fixed_1': 4159, 'fixed_2': 4556}


## Summary
The streaming + downcasting + aggregation strategy turns a ~19 GB on-disk dataset into a compact in-memory matrix (~0.35 GB for the full grid). The Parquet file is the single input for Tasks 2 and 3.